# 이커머스 멀티 모달 데이터 분석 및 RFM 리포트

UCI Online Retail 거래 데이터를 기반으로 수치형 거래 정보, 상품명 텍스트, 상품 코드/상품명에서 생성한 NumPy 이미지 배열 피처를 하나의 분석 흐름으로 처리한다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import DataAnalyzer

analyzer = DataAnalyzer(PROJECT_ROOT / 'data/ecommerce/data.csv')
df = analyzer.load_data()
df.info()
df.head()


데이터는 541,909건, 17개 컬럼으로 구성된다. 거래 기간은 2010-12-01부터 2011-12-09까지이며, RFM 계산에 사용된 고객 수는 4,338명이다. 단가 평균은 4.61, 중앙값은 2.08로 고가 상품과 비정상 입력값이 평균을 끌어올리는 오른쪽 꼬리 분포가 확인된다.

In [ ]:
df.describe(include='all').T
missing_before = df.isna().sum()
df = analyzer.handle_missing_values(numeric_strategy='group_median', group_col='category')
df = analyzer.engineer_multimodal_features(store_arrays=False)
outliers, bounds = analyzer.detect_outliers('unit_price')
df = analyzer.cap_outliers('unit_price', output_col='unit_price_capped')
rfm = analyzer.calculate_rfm()
missing_after = df.isna().sum()
missing_before, missing_after, bounds, rfm.head()


IQR 기준 단가 이상치는 39,627건(7.31%)이다. 결측치는 136534개에서 135080개로 감소했다. 상품명 단어 수, 문자열 길이, 이미지 평균, 이미지 표준편차, 엣지 강도는 반복문이 아니라 NumPy/Pandas 벡터 연산으로 산출한다.

In [ ]:
numeric_cols = ['quantity', 'unit_price', 'amount', 'name_word_count', 'image_mean', 'image_std', 'edge_strength']
numeric_summary = analyzer.summarize_numeric(numeric_cols)
corr = analyzer.correlation_matrix(['quantity', 'unit_price_capped', 'amount', 'name_word_count', 'image_mean', 'image_std', 'edge_strength'])
numeric_summary, corr


## 상관계수 해석

| 변수 쌍 | 상관계수 | 해석과 시사점 |
|---|---:|---|
| quantity-amount | 0.887 | 강한 관계가 있어 한 변수가 변할 때 다른 변수도 함께 움직일 가능성이 높다. |
| edge_strength-image_std | 0.235 | 약한 관계가 있지만 단독 설명 변수로 쓰기에는 부족하다. |
| name_word_count-image_std | 0.038 | 관계가 거의 없어 가격/매출 판단에는 다른 피처가 필요하다. |
| image_mean-name_word_count | -0.035 | 관계가 거의 없어 가격/매출 판단에는 다른 피처가 필요하다. |
| name_word_count-unit_price_capped | -0.031 | 관계가 거의 없어 가격/매출 판단에는 다른 피처가 필요하다. |

`quantity-amount`처럼 강한 상관이 있는 조합은 대량 주문 관리 지표로 쓰고, 이미지/텍스트 파생 변수는 보조 피처로 제한한다.

## 차트별 제목과 축 레이블

| 차트 | 파일 | 제목 | X축 | Y축 |
|---|---|---|---|---|
| 히스토그램 | `evidence/histogram_unit_price.png` | Unit Price Distribution After IQR Capping | Unit price capped | Transaction count |
| 박스플롯 | `evidence/boxplot_outlier_before_after.png` | Unit Price Outlier Treatment Before vs After | Treatment stage | Unit price |
| 막대그래프 | `evidence/bar_rfm_segments.png` | RFM Segment Customer Counts | RFM segment | Customer count |
| 히트맵 | `evidence/heatmap_correlation.png` | Numeric Feature Correlation Matrix | Feature | Feature |
| 산점도 | `evidence/scatter_image_mean_price.png` | Image Mean vs Unit Price | Image mean | Unit price capped |
| 라인차트 | `evidence/line_monthly_revenue.png` | Monthly Revenue Trend | Order month | Revenue |
| 보너스 히트맵 | `evidence/bonus_cohort_retention_heatmap.png` | Cohort Retention Rate | Months since first purchase | First purchase cohort |

![histogram_unit_price](../evidence/histogram_unit_price.png)

- 해석: IQR 처리 후에도 단가가 낮은 거래에 집중되어 있어 대표 단가는 평균보다 중앙값이 안정적이다.

![boxplot_outlier_before_after](../evidence/boxplot_outlier_before_after.png)

- 해석: 처리 전 극단 단가가 분포를 압축하므로 처리 후 컬럼을 비교 분석용으로 함께 보관한다.

![bar_rfm_segments](../evidence/bar_rfm_segments.png)

- 해석: Churned와 VIP가 가장 큰 의사결정 축이므로 방어 캠페인과 핵심 고객 유지 캠페인을 분리해야 한다.

![heatmap_correlation](../evidence/heatmap_correlation.png)

- 해석: 수량-금액 관계가 가장 강하고 이미지/텍스트 파생 변수의 가격 설명력은 제한적이다.

![scatter_image_mean_price](../evidence/scatter_image_mean_price.png)

- 해석: 이미지 평균 밝기만으로 단가 군집이 뚜렷하게 갈리지 않아 추가 이미지 특성이 필요하다.

![line_monthly_revenue](../evidence/line_monthly_revenue.png)

- 해석: 월별 매출 변동이 커서 세그먼트 캠페인은 시즌성과 함께 평가해야 한다.

![bonus_cohort_retention_heatmap](../evidence/bonus_cohort_retention_heatmap.png)

- 해석: 첫 구매 월별 재구매율 차이가 있어 New 고객 캠페인은 유입 코호트별로 나눠 검증한다.

## RFM 세그먼트별 특징

고객 수, 평균 최근성, 평균 빈도, 평균 구매 금액, 매출 비중을 함께 보아 세그먼트의 운영 우선순위를 정한다.

| 세그먼트 | 고객 수 | 평균 Recency | 평균 Frequency | 평균 Monetary | 매출 비중 | 특징/액션 |
|---|---:|---:|---:|---:|---:|---|
| Churned | 1,564 | 193.9일 | 1.98회 | 567.50 | 10.0% | 마지막 구매 카테고리 기반 윈백 쿠폰과 이탈 사유 설문을 병행한다. |
| VIP | 945 | 11.5일 | 11.16회 | 6,077.30 | 64.4% | 전용 멤버십과 신상품 선공개로 이탈을 막는다. |
| At Risk | 535 | 51.1일 | 1.56회 | 623.25 | 3.7% | 관심 약화 고객에게 가격/재입고 알림을 보내 반응을 측정한다. |
| Loyal | 506 | 36.4일 | 5.16회 | 1,860.17 | 10.6% | 반복 구매 카테고리 추천과 적립 혜택으로 구매 주기를 유지한다. |
| Big Spenders | 282 | 102.8일 | 2.16회 | 2,860.61 | 9.1% | 프리미엄 번들, 대량 구매 견적, 전담 상담 링크를 제공한다. |
| Regular | 279 | 15.1일 | 2.18회 | 490.32 | 1.5% | 일반 프로모션과 추천 영역으로 유지 비용을 낮춘다. |
| New | 227 | 17.7일 | 1.00회 | 275.76 | 0.7% | 첫 구매 후 7일 이내 2회차 구매 쿠폰으로 전환을 유도한다. |

## 비즈니스 인사이트

- VIP는 고객 비중 21.8%, 매출 비중 64.4%이므로 전용 혜택의 손익을 마진 데이터와 함께 검증해야 한다.
- Churned는 1,564명이고 평균 Recency가 193.9일이라 윈백 대상 규모가 가장 크지만 계절 구매자를 실제 이탈로 오분류할 수 있다.
- New는 평균 Frequency가 1.00회라 2회차 구매 전환 캠페인의 직접 대상이다.

검증에는 캠페인 노출, 클릭, 쿠폰 사용, 반품, 유입 채널, 상품 마진 데이터가 추가로 필요하다.